In [ ]:
import pandas as pd
import tensorflow as tf
import keras

from benchmark_model import BenchmarkModel, ConvBlock1D
from sklearn.metrics import classification_report, roc_auc_score

df_test = pd.read_csv('/home/jsoychak/cs/ecg/data/mitbih_test.csv')

@classmethod
    def plot_batch(cls, x, y):
        x = np.squeeze(x)
        y = np.squeeze(y)

        batch_size, window_size = x.shape
        _, depth = y.shape

        _, axes = plt.subplots(batch_size, 2, figsize=(10, batch_size * 2))
        axes = np.atleast_2d(axes)
    
        batch = zip(x, y)
        for (x_i, y_i), (ax_x, ay_y) in zip(batch, axes):
            ax_x.plot(range(window_size), x_i)
            ay_y.scatter(range(depth), y_i)
        
        plt.tight_layout()
        plt.show()

# Load test data
    df_test = pd.read_csv('/home/jsoychak/cs/ecg/data/mitbih_test.csv')
    
    x_test = df_test.to_numpy()
    x_test, y_test = x_test[:,:-1], x_test[:,-1]

    normalize_fn = dataset.get_normalize_fn
    x_test = tf.expand_dims(x_test, axis=-1)
    x_test = normalize_fn(x_test)
    y_test = tf.cast(y_test, dtype=tf.int32)
    y_test = tf.one_hot(y_test, depth=5)

    model.predict(x_test, y_test)

def prepare_data(data: pd.DataFrame):
    x = data.to_numpy()
    x, y = x[:, :-1], x[:, -1]
    x = tf.expand_dims(x, axis=-1)

    y = tf.cast(y, dtype=tf.int32)
    y = tf.one_hot(y, 5)
    
    return x, y

x_test, y_test = prepare_data(df_test)

model = keras.models.load_model(
    '/home/jsoychak/cs/ecg/src/benchmark.keras',
    custom_objects={
        'BenchmarkModel': BenchmarkModel,
        'ConvBlock1D': ConvBlock1D
    }
)

model.evaluate(x_test, y_test)

y_pred = model.predict(x_test)
y_true = y_test.numpy()

print(classification_report(y_true.argmax(axis=1), y_pred.argmax(axis=1)))

score = roc_auc_score(y_true, y_pred, average='macro', multi_class='ovr')
print('Macro AUC:', score)